In [1]:
import pandas as pd
import glob
import numpy as np

In [4]:
# Tìm file đã làm sạch
file_paths = glob.glob(r'..\processed\temp_cleaned\cleaned_yellow_tripdata_*.parquet')

file_paths.sort() 


list_of_dataframes = []

for path in file_paths:
    df_month = pd.read_parquet(path)
    list_of_dataframes.append(df_month)

# Gộp tất cả các DataFrame thành một DataFrame duy nhất
df_cleaned = pd.concat(list_of_dataframes, ignore_index=True)

# Đọc file lookup
df_lookup = pd.read_csv(r'..\raw\taxi_zone_lookup.csv')

# Chuẩn hóa VendorID
vendor_id_map = {
    1: 'Creative Mobile Technologies, LLC',
    2: 'Curb Mobility, LLC',
    6:'Myle Technologies Inc'
}
df_cleaned['VendorID'] = df_cleaned['VendorID'].map(vendor_id_map)

# # Chuẩn hóa passenger_count
# df_cleaned['passenger_count'] = df_cleaned['passenger_count'].astype(int)

# Chuẩn hóa RatecodeID
# df_cleaned['RatecodeID'] = df_cleaned['RatecodeID'].astype(int)
ratecode_id_map = {
    1.0: 'Standard rate',
    2.0: 'JFK',
    3.0: 'Newark',
    4.0: 'Nassau or Westchester',
    5.0: 'Negotiated fare',
    6.0: 'Group ride'
}
df_cleaned['RatecodeID'] = df_cleaned['RatecodeID'].map(ratecode_id_map)

# Chuẩn hóa payment_type
payment_types_map = {
    1: 'Credit card',
    2: 'Cash'
}
df_cleaned['payment_type'] = df_cleaned['payment_type'].map(payment_types_map)

# Chuẩn hóa PULocationID
df_lookup_pu = df_lookup.rename(columns={
    'LocationID': 'PULocationID',
    'Borough': 'pickup_borough',
    'Zone': 'pickup_zone'
}) # Đổi tên cột trong df_lookup để hợp nhất
df_cleaned = pd.merge(df_cleaned, df_lookup_pu, on='PULocationID', how='left') # Hợp nhất để lấy thông tin vùng đón khách

# Chuẩn hóa DOLocationID
df_lookup_do = df_lookup.rename(columns={
    'LocationID': 'DOLocationID',
    'Borough': 'dropoff_borough',
    'Zone': 'dropoff_zone'
}) # Đổi tên cột trong df_lookup để hợp nhất
df_cleaned = pd.merge(df_cleaned, df_lookup_do, on='DOLocationID', how='left') # Hợp nhất để lấy thông tin vùng trả khách


# Lưu dữ liệu
df_cleaned.to_parquet(r'..\processed\temp_cleaned\all_cleaned_yellow_tripdata_2022.parquet', index=False)

In [5]:
df_cleaned.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,congestion_surcharge,airport_fee,trip_duration_minutes,trip_speed_mph,pickup_borough,pickup_zone,service_zone_x,dropoff_borough,dropoff_zone,service_zone_y
0,"Creative Mobile Technologies, LLC",2022-01-01 00:35:40,2022-01-01 00:53:29,2.0,3.80,Standard rate,N,142,236,Credit card,...,2.5,0.0,17.816667,12.797007,Manhattan,Lincoln Square East,Yellow Zone,Manhattan,Upper East Side North,Yellow Zone
1,"Creative Mobile Technologies, LLC",2022-01-01 00:33:43,2022-01-01 00:42:07,1.0,2.10,Standard rate,N,236,42,Credit card,...,0.0,0.0,8.400000,15.000000,Manhattan,Upper East Side North,Yellow Zone,Manhattan,Central Harlem North,Boro Zone
2,"Curb Mobility, LLC",2022-01-01 00:53:21,2022-01-01 01:02:19,1.0,0.97,Standard rate,N,166,166,Credit card,...,0.0,0.0,8.966667,6.490706,Manhattan,Morningside Heights,Boro Zone,Manhattan,Morningside Heights,Boro Zone
3,"Curb Mobility, LLC",2022-01-01 00:25:21,2022-01-01 00:35:23,1.0,1.09,Standard rate,N,114,68,Cash,...,2.5,0.0,10.033333,6.518272,Manhattan,Greenwich Village South,Yellow Zone,Manhattan,East Chelsea,Yellow Zone
4,"Curb Mobility, LLC",2022-01-01 00:36:48,2022-01-01 01:14:20,1.0,4.30,Standard rate,N,68,163,Credit card,...,2.5,0.0,37.533333,6.873890,Manhattan,East Chelsea,Yellow Zone,Manhattan,Midtown North,Yellow Zone


In [ ]:
# Các cột cần dùng để tính KPI
columns_needed = [
    'tpep_pickup_datetime',
    'tpep_dropoff_datetime',
]

file_path = r'..\processed\temp_cleaned\all_cleaned_yellow_tripdata_2022.parquet'
df_kpi_dow= pd.read_parquet(file_path, columns=columns_needed)

df_kpi_dow['Trip_Duration'] = (df_kpi_dow['tpep_dropoff_datetime'] - df_kpi_dow['tpep_pickup_datetime']).dt.total_seconds() / 60

df_kpi_dow['Day_of_Week'] = df_kpi_dow['tpep_pickup_datetime'].dt.day_name()

# Tính toán P95 Trip Duration theo thứ trong tuần
kpi_dow = df_kpi_dow.groupby('Day_of_Week').agg(
    p95_trip_duration=('Trip_Duration', lambda x: x.quantile(0.95))
)

# Sắp xếp thứ trong tuần
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
kpi_dow = kpi_dow.reindex(weekday_order)

# Lưu kết quả
output = r'..\processed\kpi_dow_p95_trip_duration.csv'
kpi_dow.to_csv(output)

In [ ]:
# Các cột cần sử dụng để phân tích
columns_needed = [
    'tpep_pickup_datetime',
    'tpep_dropoff_datetime',
    'trip_distance'
]

file_path = r'..\processed\temp_cleaned\all_cleaned_yellow_tripdata_2022.parquet'
df_kpi_speed = pd.read_parquet(file_path, columns=columns_needed)

df_kpi_speed['trip_duration'] = (df_kpi_speed['tpep_dropoff_datetime'] - df_kpi_speed['tpep_pickup_datetime']).dt.total_seconds() / 3600
df_kpi_speed['speed_mph'] = df_kpi_speed['trip_distance'].divide(df_kpi_speed['trip_duration']).replace([np.inf, -np.inf], np.nan)

df_kpi_speed['hour_of_day'] = df_kpi_speed['tpep_pickup_datetime'].dt.hour

# Tính toán tốc độ trung vị (p50) theo giờ trong ngày
kpi_speed = df_kpi_speed.groupby('hour_of_day').agg(
    p50_speed_mph=('speed_mph', 'median')
)

# Lưu kết quả
output_path = r'..\processed\kpi_speed_by_hour.csv'
kpi_speed.to_csv(output_path)